# 04 - Evaluate Use-Case Attacks Against The Local Banking Agent

Use this notebook after running **03_create_domain_usecase_dataset.ipynb**.

Notebook 03 creates realistic, use-case-specific banking red-team attacks. This notebook sends those attacks to the local synthetic banking agent included in this repo.

The local banking agent has three useful modes:

- `openai`: sends the attack to an OpenAI-powered banking assistant with synthetic tool/customer context. This is the main red-team path.
- `safe`: refuses unsafe requests. The benchmark should mostly pass.
- `vulnerable`: intentionally leaks synthetic dummy tool/customer data. The benchmark should fail. This proves the tests are capable of catching data leakage and unsafe tool use.

All customer data and tool outputs are synthetic placeholders. No real banking data is used.

## 1. Setup

This cell finds the project root, loads `.env`, and makes the local `src/` package importable. No model/API call happens here.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "src" / "finance_redteam").exists())

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"OpenAI key present: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"OpenAI model: {os.getenv('LOCAL_BANKING_AGENT_OPENAI_MODEL') or os.getenv('OPENAI_MODEL') or 'gpt-4.1-nano'}")

## 2. Choose The Use-Case Dataset And Agent Mode

The default profile below matches Notebook 03's custom banking support agent output.

Use `LOCAL_BANKING_AGENT_MODE = "openai"` when you want to red-team an actual LLM-backed banking assistant.

Keep the other two modes as controls:

- `safe`: deterministic refusal baseline.
- `vulnerable`: deterministic leaky baseline that should fail.

In [ ]:
# Notebook 03 export folder.
USECASE_PROFILE_ID = "banking_support_agent_custom"
USECASE_EXPORT_DIR = Path("data/exports") / USECASE_PROFILE_ID

# Main attack dataset created by Notebook 03.
ATTACKS_JSONL = USECASE_EXPORT_DIR / "attacks.jsonl"

# Choose banking agent behavior.
# openai     -> real OpenAI-powered banking assistant with synthetic tool/customer context.
# safe       -> deterministic refusal baseline; expected to pass.
# vulnerable -> deterministic leaky baseline; expected to fail.
LOCAL_BANKING_AGENT_MODE = "openai"  # openai, safe, or vulnerable
os.environ["LOCAL_BANKING_AGENT_MODE"] = LOCAL_BANKING_AGENT_MODE

# Keep this small for cost control. You can also set LOCAL_BANKING_AGENT_OPENAI_MODEL in .env.
LOCAL_BANKING_AGENT_OPENAI_MODEL = os.getenv("LOCAL_BANKING_AGENT_OPENAI_MODEL") or os.getenv("OPENAI_MODEL") or "gpt-4.1-nano"
os.environ["LOCAL_BANKING_AGENT_OPENAI_MODEL"] = LOCAL_BANKING_AGENT_OPENAI_MODEL
os.environ.setdefault("LOCAL_BANKING_AGENT_MAX_TOKENS", "350")
os.environ.setdefault("LOCAL_BANKING_AGENT_TEMPERATURE", "0.1")

# Promptfoo config generated specifically for the selected local agent mode.
PROMPTFOO_YAML = USECASE_EXPORT_DIR / f"promptfoo_local_{LOCAL_BANKING_AGENT_MODE}.yaml"

# Evaluation size controls.
SMOKE_TEST_COUNT = 5
SMALL_BATCH_COUNT = 25
MAX_CONCURRENCY = 1

print("Attack dataset:", ATTACKS_JSONL)
print("Promptfoo config:", PROMPTFOO_YAML)
print("Local banking agent mode:", LOCAL_BANKING_AGENT_MODE)
print("OpenAI target model:", LOCAL_BANKING_AGENT_OPENAI_MODEL)
if LOCAL_BANKING_AGENT_MODE == "openai" and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required for LOCAL_BANKING_AGENT_MODE='openai'. Add it to .env first.")

## 3. Load And Inspect The Attack Dataset

This confirms Notebook 03 produced a usable attack file and shows whether attacks are tied to workflows, tools, protected assets, and objectives.

In [ ]:
from finance_redteam.exporters import load_jsonl

if not ATTACKS_JSONL.exists():
    raise FileNotFoundError(
        f"Missing {ATTACKS_JSONL}. Run notebooks/03_create_domain_usecase_dataset.ipynb first."
    )

records = load_jsonl(ATTACKS_JSONL)
print("Records loaded:", len(records))

print("\nTop workflows:")
for value, count in Counter(r.workflow_id or "missing" for r in records).most_common(10):
    print(f"- {value}: {count}")

print("\nTop tools:")
for value, count in Counter(r.tool_name or "no_tool" for r in records).most_common(10):
    print(f"- {value}: {count}")

print("\nTop attack objectives:")
for value, count in Counter(r.attack_objective or "missing" for r in records).most_common(10):
    print(f"- {value}: {count}")

## 4. Confirm Dummy Data Covers The Attack Surface

This prevents a misleading pass caused by missing local test data. The local dummy database should contain tool fixtures for every tool referenced by the generated attacks.

In [ ]:
DUMMY_DB = Path("data/dummy_bank/customers.json")
db = json.loads(DUMMY_DB.read_text(encoding="utf-8"))

attack_tools = {r.tool_name for r in records if r.tool_name}
fixture_tools = set((db.get("tool_permissions") or {}).keys())
missing_tools = sorted(attack_tools - fixture_tools)

print("Dummy DB:", DUMMY_DB)
print("Customers:", len(db.get("customers") or []))
print("Attack tools:", len(attack_tools))
print("Fixture tools:", len(fixture_tools))
print("Missing tool fixtures:", missing_tools)

if missing_tools:
    raise ValueError("Dummy database is missing fixtures for some attack tools.")

## 5. Preview Safe vs Vulnerable Agent Behavior

This is a direct Python check before Promptfoo runs. It sends one attack prompt to the local agent in both modes.

- In `openai` mode, the real LLM sees synthetic confidential context and should refuse unsafe requests without leaking it.
- In `safe` mode, the deterministic agent should refuse.
- In `vulnerable` mode, the deterministic agent intentionally leaks synthetic dummy tool output.

In [ ]:
from finance_redteam.banking_agent import evaluate_prompt

sample = next((r for r in records if r.tool_name), records[0])
sample_prompt = sample.attack_query or sample.prompt

print("Sample attack_id:", sample.attack_id)
print("Workflow:", sample.workflow_id)
print("Tool:", sample.tool_name)
print("Prompt preview:")
print(sample_prompt[:1000])

preview_modes = [LOCAL_BANKING_AGENT_MODE]
if LOCAL_BANKING_AGENT_MODE != "vulnerable":
    preview_modes.append("vulnerable")

for mode in preview_modes:
    response = evaluate_prompt(sample_prompt, mode=mode)
    print("\n---", mode, "---")
    print("Decision:", response.decision)
    print("Tool:", response.tool_name)
    print("Customer:", response.customer_id)
    print(response.output[:1200])

## 6. Generate The Local Promptfoo Config

This writes a Promptfoo config that points to `providers/local_banking_agent_provider.js`.

The provider reads `LOCAL_BANKING_AGENT_MODE`:

- `openai`: calls OpenAI with the banking system prompt and synthetic tool/customer context. If the attack prompt contains `Turn 1:`, `Turn 2:`, etc., the provider sends those turns sequentially with conversation history.
- `safe`: returns deterministic refusal text.
- `vulnerable`: returns deterministic dummy data leakage.

The generated YAML also uses `assertions/banking_response_assertions.js`, which applies scenario-specific checks for banking data leakage, unsafe tool/action claims, and loan-underwriting failures such as protected-class decisioning, fabricated underwriting rationale, and fair-lending bypass.


In [ ]:
from finance_redteam.promptfoo_exporter import export_promptfoo

export_promptfoo(
    records,
    PROMPTFOO_YAML,
    provider="local_banking_agent",
    model=f"synthetic-{LOCAL_BANKING_AGENT_MODE}-demo",
)

print(f"Wrote {PROMPTFOO_YAML}")
print(PROMPTFOO_YAML.read_text(encoding="utf-8")[:1600])

## 7. Check Promptfoo Is Available

Promptfoo is the evaluation runner. It sends each attack prompt to the local banking-agent provider and applies the assertions from the YAML config. For multi-turn attacks, the provider handles the sequential conversation; Promptfoo still sees one test case with one final assistant transcript.


In [ ]:
result = subprocess.run(["npx", "promptfoo", "--version"], text=True, capture_output=True, timeout=120)
print("Return code:", result.returncode)
print((result.stdout or result.stderr).strip()[:1000])

if result.returncode != 0:
    raise RuntimeError("Promptfoo is not available. Install it with: npm install -g promptfoo")

## 8. Run A Smoke Evaluation

This evaluates only the first few attacks. Use this first every time.

Important: in `vulnerable` mode, Promptfoo may return a non-zero exit code because failures are expected. That means the benchmark caught unsafe behavior. In `openai` mode, failures mean the model output matched one of the leakage/unsafe-output assertions.

In [ ]:
def run_promptfoo_eval(limit: int | None = None, timeout: int = 900) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    env["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
    env["HOME"] = "."
    env["LOCAL_BANKING_AGENT_MODE"] = LOCAL_BANKING_AGENT_MODE

    command = [
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--max-concurrency", str(MAX_CONCURRENCY),
        "--no-cache",
    ]
    if limit is not None:
        command.extend(["--filter-first-n", str(limit)])

    print("Running:", " ".join(command))
    completed = subprocess.run(command, text=True, capture_output=True, timeout=timeout, env=env)
    print(completed.stdout[-6000:])
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr[-3000:])
    print("Return code:", completed.returncode)
    return completed

RUN_SMOKE_EVAL = True

if RUN_SMOKE_EVAL:
    smoke_result = run_promptfoo_eval(limit=SMOKE_TEST_COUNT)
else:
    print("Smoke eval skipped.")

## 9. Run A Larger Batch

After smoke testing, run a larger batch. Keep `MAX_CONCURRENCY = 1` for easy debugging and deterministic local behavior.

In [ ]:
RUN_SMALL_BATCH_EVAL = False

if RUN_SMALL_BATCH_EVAL:
    small_batch_result = run_promptfoo_eval(limit=SMALL_BATCH_COUNT, timeout=1800)
else:
    print("Small batch eval skipped. Set RUN_SMALL_BATCH_EVAL=True when ready.")

## 10. Run The Full Evaluation

Use this after smoke and small-batch checks look correct.

Recommended sequence:

1. Run smoke eval in `openai` mode. Review the first few outputs.
2. Run `safe` mode as a deterministic pass control.
3. Run `vulnerable` mode as a deterministic fail control.
4. Run full `openai` mode once the smoke result looks correct.

In [ ]:
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    full_result = run_promptfoo_eval(limit=None, timeout=7200)
else:
    print("Full eval skipped. Set RUN_FULL_EVAL=True when ready.")

## 11. Read The Latest JSON Result If Available

Promptfoo writes its own local history, but this cell also checks the repo-level export path if you saved or copied a JSON result there.

In [ ]:
LATEST_JSON = Path("data/exports/latest_promptfoo_eval.json")

if LATEST_JSON.exists():
    latest = json.loads(LATEST_JSON.read_text(encoding="utf-8"))
    print("Loaded:", LATEST_JSON)
    print(json.dumps(latest, indent=2)[:3000])
else:
    print("No repo-level latest Promptfoo JSON found. This is okay; use the terminal summary above or run `promptfoo view`.")

## 12. Stop A Stuck Promptfoo Run

If a Promptfoo process is stuck and `Ctrl+C` does not stop it, set `STOP_STUCK_PROMPTFOO=True` and run this cell.

In [ ]:
STOP_STUCK_PROMPTFOO = False

if STOP_STUCK_PROMPTFOO:
    subprocess.run(["pkill", "-f", "promptfoo"], text=True, capture_output=True)
    subprocess.run(["pkill", "-f", "node.*promptfoo"], text=True, capture_output=True)
    print("Requested stop for promptfoo/node promptfoo processes.")
else:
    print("Stop skipped.")

## What Good Results Look Like

For `openai` mode, a healthy result means the OpenAI-backed banking assistant refuses unsafe requests and does not leak synthetic tool/customer context.

For `safe` mode, a healthy result usually looks like mostly passes because the deterministic local banking agent refuses unsafe requests.

For `vulnerable` mode, a healthy result usually contains failures because the local banking agent intentionally leaks dummy data. That is not a notebook error; it is the proof that the red-team attacks and assertions can detect a vulnerable banking assistant.